In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error     # метрики
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
pd.set_option('display.max_columns', 100)  # показать до 100 колонок

In [2]:
import sys, pandas as pd, sklearn
print(sys.executable)
print(pd.__version__, sklearn.__version__)

ModuleNotFoundError: No module named 'pandas'

In [157]:
df = pd.read_csv('bank.csv',sep=';')

In [158]:
df.head()

,age,job,marital,education,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,default
0,58,management,married,tertiary,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no,no
1,44,technician,single,secondary,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no,no
2,33,entrepreneur,married,secondary,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no,no
3,47,blue-collar,married,unknown,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no,no
4,33,unknown,single,unknown,1,no,no,unknown,5,may,198,1,-1,0,unknown,no,no


In [159]:
# Создадим экземпляр LabelEncoder
label_encoder = LabelEncoder()

# Обучим LabelEncoder на столбце 'цвет' и преобразуем его
df['housing'] = label_encoder.fit_transform(df['housing'])
df['loan'] = label_encoder.fit_transform(df['loan'])
df['deposit'] = label_encoder.fit_transform(df['deposit'])
df['default'] = label_encoder.fit_transform(df['default'])

df.head()

,age,job,marital,education,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,default
0,58,management,married,tertiary,2143,1,0,unknown,5,may,261,1,-1,0,unknown,0,0
1,44,technician,single,secondary,29,1,0,unknown,5,may,151,1,-1,0,unknown,0,0
2,33,entrepreneur,married,secondary,2,1,1,unknown,5,may,76,1,-1,0,unknown,0,0
3,47,blue-collar,married,unknown,1506,1,0,unknown,5,may,92,1,-1,0,unknown,0,0
4,33,unknown,single,unknown,1,0,0,unknown,5,may,198,1,-1,0,unknown,0,0


In [162]:
# Создадим экземпляр OneHotEncoder
# sparse=False для возврата numpy массива, а не sparse matrix,
# handle_unknown='ignore' обрабатывает неизвестные категории,которые могут появится позже
# drop='first' удаляет один столбец для избежания мультиколлинеарности
# pd.set_option('display.max_colums', 100) # снимает ограничение просмотра столбцов по ширине экрана
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # , drop='first') удаляет первый столбец из 'yes, no'
# Обучим OneHotEncoder на столбце 'цвет' и преобразуем данные
ft= encoder.fit_transform(df[['default']])
# Создадим DataFrame из результатов One-Hot Encoding
onehot_df = pd.DataFrame(ft, columns=encoder.get_feature_names_out(['default']))
# Объединим исходный DataFrame с One-Hot Encoding
df = pd.concat([df, onehot_df], axis=1)
df = df.drop(columns=['default_1','default_0', 'poutcome', 'job','marital','education','contact','month'])
#df = df.drop('default_no', axis=1)
df

,age,balance,housing,loan,day,duration,campaign,pdays,previous,deposit,default
0,58,2143,1,0,5,261,1,-1,0,0,0
1,44,29,1,0,5,151,1,-1,0,0,0
2,33,2,1,1,5,76,1,-1,0,0,0
3,47,1506,1,0,5,92,1,-1,0,0,0
4,33,1,0,0,5,198,1,-1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
45206,51,825,0,0,17,977,3,-1,0,1,0
45207,71,1729,0,0,17,456,2,-1,0,1,0
45208,72,5715,0,0,17,1127,5,184,3,1,0
45209,57,668,0,0,17,508,4,-1,0,0,0


In [ ]:
# Создадим экземпляр StandardScaler
scaler = StandardScaler()

# Обучим StandardScaler на данных и преобразуем их
df_scaled = scaler.fit_transform(df)

# Преобразуем в DataFrame
df_scaled = pd.DataFrame(df_scaled, columns=df.columns)



In [170]:
df_scaled

,age,balance,housing,loan,day,duration,campaign,pdays,previous,deposit,default
0,1.606965,0.256419,0.893915,-0.436803,-1.298476,0.011016,-0.569351,-0.411453,-0.251940,-0.363983,-0.13549
1,0.288529,-0.437895,0.893915,-0.436803,-1.298476,-0.416127,-0.569351,-0.411453,-0.251940,-0.363983,-0.13549
2,-0.747384,-0.446762,0.893915,2.289359,-1.298476,-0.707361,-0.569351,-0.411453,-0.251940,-0.363983,-0.13549
3,0.571051,0.047205,0.893915,-0.436803,-1.298476,-0.645231,-0.569351,-0.411453,-0.251940,-0.363983,-0.13549
4,-0.747384,-0.447091,-1.118674,-0.436803,-1.298476,-0.233620,-0.569351,-0.411453,-0.251940,-0.363983,-0.13549
...,...,...,...,...,...,...,...,...,...,...,...
45206,0.947747,-0.176460,-1.118674,-0.436803,0.143418,2.791329,0.076230,-0.411453,-0.251940,2.747384,-0.13549
45207,2.831227,0.120447,-1.118674,-0.436803,0.143418,0.768224,-0.246560,-0.411453,-0.251940,2.747384,-0.13549
45208,2.925401,1.429593,-1.118674,-0.436803,0.143418,3.373797,0.721811,1.436189,1.050473,2.747384,-0.13549
45209,1.512791,-0.228024,-1.118674,-0.436803,0.143418,0.970146,0.399020,-0.411453,-0.251940,-0.363983,-0.13549


In [174]:
x = df_scaled.drop(['loan'], axis=1)
y = df_scaled['loan']

In [175]:
x.shape

(45211, 10)

In [177]:
y.shape

(45211,)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

# смотрим как разбилось
x_train.shape

x_test.shape

(13564, 10)